## 🎯 Learning Objectives
* Demonstrate comprehensive understanding of multi-agent system design principles.
* Implement advanced intent routing mechanisms for diverse user requests.
* Integrate and manage multiple sub-crews for complex, multi-stage tasks.
* Design and implement robust human approval gates within agentic workflows.
* Apply learned concepts to build a scalable and adaptable multi-service booking system.
* Evaluate and debug multi-agent system interactions and tool usage.


# Final Assessment: Multi-Service Booking System Extension

Welcome to the final assessment for PRJ-01: Multi-Agent AI System for Hotel Reservations! This assessment is designed to consolidate your learning and challenge you to apply the advanced concepts and techniques covered throughout the course.

Over the past weeks, you've mastered the art of building sophisticated multi-agent systems, focusing on intent routing, sub-crew orchestration, and critical human approval gates. You've built a robust system capable of handling hotel reservations, demonstrating how autonomous agents can collaborate to fulfill complex user requests.

This final assessment will push you to extend your existing knowledge to a new, yet related, domain. You will be tasked with expanding the capabilities of your booking system to handle not just hotel reservations, but also flight bookings and combined flight-hotel package deals. This requires a deeper dive into intent classification, dynamic sub-crew activation, and careful management of multi-step booking processes, all while maintaining the crucial human oversight you've implemented.

**What you will demonstrate:**
*   **Advanced Intent Routing:** Your ability to accurately classify diverse user intents (hotel, flight, package) and direct them to the appropriate sub-crews.
*   **Sub-Crew Orchestration:** Your skill in designing and integrating new sub-crews (e.g., for flight bookings) and orchestrating them for complex package deals.
*   **Human-in-the-Loop:** Your continued proficiency in embedding human approval gates at critical junctures to ensure accuracy and user satisfaction.
*   **Tool Integration:** Your capacity to define and integrate new tools for different booking services.
*   **Robustness and Error Handling:** Your system's ability to gracefully handle incomplete information and guide the user towards successful completion.

This assessment is your opportunity to showcase your expertise as an AI builder ready to tackle real-world automation challenges. Good luck!


## Review Questions

Before diving into the coding challenge, take a moment to reflect on the core concepts of the course. Answer the following questions to solidify your understanding:

1.  **Intent Router's Role:** What is the primary architectural advantage of using an explicit intent router at the entry point of a multi-agent system, especially when dealing with diverse user requests? How does it contribute to scalability and maintainability?
2.  **Human Approval Gates:** Describe a critical scenario in a booking system (hotel, flight, or car rental) where a human approval gate is absolutely essential. Explain the potential risks of *not* having one and how the gate mitigates these risks.
3.  **Sub-Crew Collaboration:** When designing a multi-agent sub-crew (e.g., for hotel booking), what strategies can you employ to ensure agents collaborate effectively without redundant work or conflicting actions? Discuss the role of shared context and task delegation.
4.  **LLM vs. Rule-Based Routing:** Compare and contrast an LLM-based intent router with a purely rule-based intent router. In what situations would you prefer one over the other for a booking system, and why?
5.  **State Management:** How is 'state' or 'memory' managed across different agents and tasks within a CrewAI-based multi-agent system? Provide an example of how an agent might pass crucial information (e.g., booking details) to another agent for a subsequent task.
6.  **Handling Ambiguity:** A user says, "I need a place to stay next month." How would your multi-agent system, particularly the intent router and initial agents, handle such an ambiguous request? What mechanisms would be in place to gather more information?
7.  **Deployment Considerations:** Beyond the code, what are the key considerations for deploying a multi-agent booking system like the one you've built into a production environment in 2026? Think about scalability, security, monitoring, and user experience.
8.  **Tool Design:** When designing tools for agents, what are the best practices to ensure they are effective, reliable, and safe? Provide an example of a well-designed tool signature and its purpose.


## Capstone Coding Problem: Multi-Service Booking System

**Scenario:**
Your successful hotel reservation system has caught the attention of a major travel agency. They want to expand its capabilities to offer a full suite of travel booking services. Your task is to evolve your existing multi-agent system into a comprehensive **Multi-Service Booking System** capable of handling:

1.  **Hotel Reservations** (as previously implemented).
2.  **Flight Reservations**.
3.  **Flight & Hotel Package Deals**.

**Core Requirements:**

1.  **Enhanced Intent Router:** Modify your `IntentRouterAgent` to accurately classify user requests into one of three primary intents:
    *   `hotel_booking`
    *   `flight_booking`
    *   `package_booking` (for combined flight and hotel requests)
    The router should be robust enough to handle variations in user phrasing.

2.  **New Flight Booking Sub-Crew:** Create a new `FlightBookingCrew` consisting of agents and tools specifically designed for flight reservations. This crew should be able to:
    *   Search for flights based on origin, destination, dates, and number of passengers.
    *   Present flight options to the user (or a summary).
    *   Facilitate the booking of a selected flight.
    *   **Crucially, include a human approval gate** before finalizing any flight booking.

3.  **Package Booking Sub-Crew:** Develop a `PackageBookingCrew` that orchestrates both flight and hotel bookings. This crew should:
    *   Understand user requests for combined flight and hotel packages.
    *   Leverage the capabilities of both the `FlightBookingCrew` and the `HotelBookingCrew` (or their underlying agents/tools) to find and book suitable options.
    *   Ensure a cohesive experience, potentially coordinating dates and locations.
    *   **Include a human approval gate** for the entire package deal before final confirmation.

4.  **Mock Tools:** Since we don't have live APIs, create simple Python functions to act as mock tools for:
    *   `search_flights(origin, destination, departure_date, return_date, passengers)`
    *   `book_flight(flight_id, passenger_details)`
    *   (Assume `search_hotels` and `book_hotel` from previous lessons are available or create simple mocks).

5.  **Robustness:** The system should be able to ask clarifying questions if initial user input is incomplete for any booking type.

**Deliverables:**
*   A single Python script or Jupyter Notebook containing the complete multi-agent system.
*   Clear definitions for all agents, tasks, and crews.
*   Well-commented code explaining the logic.
*   Demonstration of the system handling at least one scenario for each intent type (hotel, flight, package) using example prompts.

**Evaluation Criteria:**
*   **Correctness:** Does the system correctly route intents and execute bookings?
*   **Completeness:** Are all requirements met, including human approval gates and mock tools?
*   **Design:** Is the agent and crew architecture logical, modular, and extensible?
*   **Code Quality:** Is the code clean, readable, and well-commented?
*   **Robustness:** Does the system handle incomplete information gracefully?

Good luck, and demonstrate your mastery of agentic AI!


In [ ]:
import os
from crewai import Agent, Task, Crew, Process
from crewai_tools import Tool
from langchain_openai import ChatOpenAI

# --- Configuration --- #
# Set your OpenAI API key here or as an environment variable
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

# Initialize the LLM for all agents
# Using a smaller, cost-effective model for demonstration
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.7)

# --- Mock Tools (Placeholder) --- #
# Assume these are available from previous lessons or define simple mocks
class HotelTools:
    @Tool("Search Hotels")
    def search_hotels(location: str, check_in_date: str, check_out_date: str, guests: int) -> str:
        """Searches for available hotels based on location, dates, and number of guests."""
        print(f"\n--- Searching hotels in {location} from {check_in_date} to {check_out_date} for {guests} guests ---")
        # Mock data
        if "Paris" in location:
            return "Found Hotel Eiffel (ID: H101) - $200/night, Hotel Louvre (ID: H102) - $250/night."
        elif "London" in location:
            return "Found The Gherkin Hotel (ID: H201) - $180/night, Big Ben Inn (ID: H202) - $220/night."
        else:
            return "No hotels found for the specified criteria."

    @Tool("Book Hotel")
    def book_hotel(hotel_id: str, user_details: str) -> str:
        """Books a hotel with the given ID for the specified user details."""
        print(f"\n--- Attempting to book hotel {hotel_id} for {user_details} ---")
        # Mock booking success
        return f"Hotel {hotel_id} successfully booked for {user_details}. Confirmation: HBOOK-12345."

class FlightTools:
    @Tool("Search Flights")
    def search_flights(origin: str, destination: str, departure_date: str, return_date: str = None, passengers: int = 1) -> str:
        """Searches for available flights based on origin, destination, dates, and number of passengers."""
        print(f"\n--- Searching flights from {origin} to {destination} on {departure_date} (Return: {return_date}) for {passengers} passengers ---")
        # Mock data
        if "London" in origin and "Paris" in destination:
            return f"Found Flight BA201 (ID: F101) - {departure_date} 09:00, $150. Flight AF105 (ID: F102) - {departure_date} 14:00, $180."
        elif "New York" in origin and "London" in destination:
            return f"Found Flight UA001 (ID: F201) - {departure_date} 18:00, $600. Flight VS005 (ID: F202) - {departure_date} 20:00, $650."
        else:
            return "No flights found for the specified criteria."

    @Tool("Book Flight")
    def book_flight(flight_id: str, passenger_details: str) -> str:
        """Books a flight with the given ID for the specified passenger details."""
        print(f"\n--- Attempting to book flight {flight_id} for {passenger_details} ---")
        # Mock booking success
        return f"Flight {flight_id} successfully booked for {passenger_details}. Confirmation: FBOOK-67890."

# Instantiate tools
hotel_tools = HotelTools()
flight_tools = FlightTools()

# --- Agents --- #

# Intent Router Agent
intent_router_agent = Agent(
    role='Intent Router',
    goal='Direct user requests to the correct booking sub-crew (Hotel, Flight, or Package).',
    backstory="""You are an expert AI assistant specializing in understanding user intent for travel bookings.
    Your primary role is to analyze the user's request and determine if it's for a hotel, a flight, or a combined package.
    You must be precise and ensure the request is routed to the most appropriate specialized agent or crew.
    If the intent is unclear, you should ask clarifying questions to determine the correct path.
    """,
    verbose=True,
    allow_delegation=False,
    llm=llm
)

# Hotel Booking Agents
hotel_search_agent = Agent(
    role='Hotel Search Specialist',
    goal='Find the best hotel options based on user criteria.',
    backstory="""You are a meticulous hotel search expert. You use advanced search tools to find hotels that perfectly match the user's location, dates, and guest count.
    You provide clear, concise options for the booking agent.
    """,
    tools=[hotel_tools.search_hotels],
    verbose=True,
    allow_delegation=True,
    llm=llm
)

hotel_booking_agent = Agent(
    role='Hotel Booking Agent',
    goal='Confirm and finalize hotel reservations after human approval.',
    backstory="""You are a diligent hotel booking agent. Once hotel options are identified and human approval is given,
    you use the booking tool to secure the reservation and provide confirmation details.
    You are responsible for ensuring all details are correct before booking.
    """,
    tools=[hotel_tools.book_hotel],
    verbose=True,
    allow_delegation=False,
    llm=llm
)

# Flight Booking Agents (NEW)
flight_search_agent = Agent(
    role='Flight Search Specialist',
    goal='Find the best flight options based on user criteria.',
    backstory="""You are an expert in flight search, capable of finding optimal routes, dates, and prices.
    You use flight search tools to present clear options for the booking agent.
    """,
    tools=[flight_tools.search_flights],
    verbose=True,
    allow_delegation=True,
    llm=llm
)

flight_booking_agent = Agent(
    role='Flight Booking Agent',
    goal='Confirm and finalize flight reservations after human approval.',
    backstory="""You are a meticulous flight booking agent. Once flight options are identified and human approval is given,
    you use the booking tool to secure the reservation and provide confirmation details.
    You are responsible for ensuring all details are correct before booking.
    """,
    tools=[flight_tools.book_flight],
    verbose=True,
    allow_delegation=False,
    llm=llm
)

# Package Booking Agent (NEW)
package_coordinator_agent = Agent(
    role='Package Booking Coordinator',
    goal='Orchestrate combined flight and hotel bookings, ensuring seamless package deals.',
    backstory="""You are the ultimate travel package expert. You coordinate between flight and hotel search/booking agents
    to create comprehensive travel packages. You ensure all components align and present a unified package for human approval.
    You are responsible for managing the entire package booking lifecycle.
    """,
    verbose=True,
    allow_delegation=True, # Can delegate to search agents
    llm=llm
)

# --- Tasks --- #

# Intent Routing Task
route_intent_task = Task(
    description="""Analyze the user's request: '{user_request}'.
    Determine if the user wants to book a 'hotel', a 'flight', or a 'package' (flight + hotel).
    Output ONLY the determined intent as a single word: 'hotel', 'flight', or 'package'.
    If the intent is unclear, output 'clarify'.
    Example: 'I need a hotel in Paris' -> 'hotel'
    Example: 'Book me a flight to London' -> 'flight'
    Example: 'I want a trip to Rome with flight and hotel' -> 'package'
    Example: 'Tell me about travel' -> 'clarify'
    """,
    agent=intent_router_agent,
    expected_output="A single word: 'hotel', 'flight', 'package', or 'clarify'."
)

# Hotel Booking Tasks
hotel_search_task = Task(
    description="""Search for hotels based on the user's request: '{user_request}'.
    Extract location, check-in date, check-out date, and number of guests.
    If any information is missing, ask clarifying questions to the user.
    Provide a summary of available options, including hotel IDs and prices.
    """,
    agent=hotel_search_agent,
    expected_output="A list of available hotel options with IDs and prices, or a request for more information."
)

hotel_approval_task = Task(
    description="""Review the proposed hotel options and ask for human approval.
    Present the best option(s) found by the Hotel Search Specialist.
    Wait for human input (e.g., 'approve H101' or 'reject').
    If approved, pass the chosen hotel ID and user details for booking.
    """,
    agent=hotel_booking_agent, # This agent will handle the human input
    human_input=True,
    expected_output="The chosen hotel ID and user details for booking, or a rejection message."
)

hotel_booking_task = Task(
    description="""Book the approved hotel using the provided hotel ID and user details.
    Generate a confirmation message including the booking ID.
    """,
    agent=hotel_booking_agent,
    expected_output="A confirmation message for the hotel booking."
)

# Flight Booking Tasks (NEW)
flight_search_task = Task(
    description="""Search for flights based on the user's request: '{user_request}'.
    Extract origin, destination, departure date, return date (if applicable), and number of passengers.
    If any information is missing, ask clarifying questions to the user.
    Provide a summary of available flight options, including flight IDs and prices.
    """,
    agent=flight_search_agent,
    expected_output="A list of available flight options with IDs and prices, or a request for more information."
)

flight_approval_task = Task(
    description="""Review the proposed flight options and ask for human approval.
    Present the best option(s) found by the Flight Search Specialist.
    Wait for human input (e.g., 'approve F101' or 'reject').
    If approved, pass the chosen flight ID and passenger details for booking.
    """,
    agent=flight_booking_agent,
    human_input=True,
    expected_output="The chosen flight ID and passenger details for booking, or a rejection message."
)

flight_booking_task = Task(
    description="""Book the approved flight using the provided flight ID and passenger details.
    Generate a confirmation message including the booking ID.
    """,
    agent=flight_booking_agent,
    expected_output="A confirmation message for the flight booking."
)

# Package Booking Tasks (NEW)
package_search_and_propose_task = Task(
    description="""Coordinate the search for both flights and hotels based on the user's package request: '{user_request}'.
    Extract all necessary details for both flight (origin, destination, dates, passengers) and hotel (location, dates, guests).
    Use the Flight Search Specialist and Hotel Search Specialist to find suitable options.
    Propose a combined package deal, including flight and hotel details, for human review.
    If any information is missing, ask clarifying questions.
    """,
    agent=package_coordinator_agent,
    expected_output="A proposed flight and hotel package summary, or a request for more information."
)

package_approval_task = Task(
    description="""Review the proposed flight and hotel package and ask for human approval.
    Present the complete package details found by the Package Booking Coordinator.
    Wait for human input (e.g., 'approve package' or 'reject').
    If approved, pass the chosen flight ID, hotel ID, and user details for final booking.
    """,
    agent=package_coordinator_agent,
    human_input=True,
    expected_output="The chosen flight ID, hotel ID, and user details for booking, or a rejection message."
)

package_final_booking_task = Task(
    description="""Finalize the approved flight and hotel package booking.
    This involves booking both the flight and the hotel sequentially or in parallel.
    Generate a comprehensive confirmation message for the entire package.
    """,
    agent=package_coordinator_agent,
    expected_output="A comprehensive confirmation message for the flight and hotel package booking."
)

# --- Crews --- #

hotel_booking_crew = Crew(
    agents=[hotel_search_agent, hotel_booking_agent],
    tasks=[hotel_search_task, hotel_approval_task, hotel_booking_task],
    process=Process.sequential,
    verbose=True
)

flight_booking_crew = Crew(
    agents=[flight_search_agent, flight_booking_agent],
    tasks=[flight_search_task, flight_approval_task, flight_booking_task],
    process=Process.sequential,
    verbose=True
)

package_booking_crew = Crew(
    agents=[package_coordinator_agent, hotel_search_agent, hotel_booking_agent, flight_search_agent, flight_booking_agent], # Include all necessary agents
    tasks=[
        package_search_and_propose_task,
        package_approval_task,
        package_final_booking_task
    ],
    process=Process.sequential,
    verbose=True
)

# --- Main Router Crew --- #

class MultiServiceBookingSystem:
    def __init__(self):
        self.router_crew = Crew(
            agents=[intent_router_agent],
            tasks=[route_intent_task],
            process=Process.sequential,
            verbose=True
        )

    def run(self, user_request: str):
        print(f"\n--- User Request: {user_request} ---")
        intent_result = self.router_crew.kickoff(inputs={'user_request': user_request})
        intent = intent_result.strip().lower()

        print(f"\n--- Detected Intent: {intent} ---")

        if intent == 'hotel':
            print("\n--- Activating Hotel Booking Crew ---")
            result = hotel_booking_crew.kickoff(inputs={'user_request': user_request})
        elif intent == 'flight':
            print("\n--- Activating Flight Booking Crew ---")
            result = flight_booking_crew.kickoff(inputs={'user_request': user_request})
        elif intent == 'package':
            print("\n--- Activating Package Booking Crew ---")
            result = package_booking_crew.kickoff(inputs={'user_request': user_request})
        elif intent == 'clarify':
            result = "I'm sorry, I couldn't clearly understand your request. Could you please provide more details about what you'd like to book (hotel, flight, or a package)?"
        else:
            result = "An unexpected error occurred in intent routing."

        print("\n--- Final System Response ---")
        print(result)
        return result

# --- Example Usage --- #
if __name__ == "__main__":
    booking_system = MultiServiceBookingSystem()

    # Example 1: Hotel Booking
    # booking_system.run("I need a hotel in Paris for 2 adults from 2026-07-10 to 2026-07-15.")

    # Example 2: Flight Booking
    # booking_system.run("Book me a flight from London to Paris for one person on 2026-08-20, returning on 2026-08-25.")

    # Example 3: Package Booking
    # booking_system.run("I want a package trip to London from New York for 2 people, departing 2026-09-01 and returning 2026-09-07. Also need a hotel there.")

    # Example 4: Unclear Request
    # booking_system.run("Tell me about travel options.")

    print("\n--- Uncomment the example usage lines to run specific scenarios. ---")
    print("--- Remember to interact with the human approval gates when prompted. ---")


In [ ]:
import os
from crewai import Agent, Task, Crew, Process
from crewai_tools import Tool
from langchain_openai import ChatOpenAI

# --- Configuration --- #
# Set your OpenAI API key here or as an environment variable
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

# Initialize the LLM for all agents
# Using a smaller, cost-effective model for demonstration
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.7)

# --- Mock Tools --- #
# These tools simulate interactions with external APIs for hotel and flight services.
# In a real-world scenario, these would connect to actual booking platforms.

class HotelTools:
    @Tool("Search Hotels")
    def search_hotels(location: str, check_in_date: str, check_out_date: str, guests: int) -> str:
        """Searches for available hotels based on location, dates, and number of guests.
        Returns a string summary of found hotels including their IDs and prices.
        Example: 'Found Hotel Eiffel (ID: H101) - $200/night, Hotel Louvre (ID: H102) - $250/night.'
        """
        print(f"\n[Tool Call] Searching hotels in {location} from {check_in_date} to {check_out_date} for {guests} guests")
        # Mock data based on location
        if "Paris" in location:
            return f"Found Hotel Eiffel (ID: H101) - $200/night, Hotel Louvre (ID: H102) - $250/night for {guests} guests from {check_in_date} to {check_out_date}."
        elif "London" in location:
            return f"Found The Gherkin Hotel (ID: H201) - $180/night, Big Ben Inn (ID: H202) - $220/night for {guests} guests from {check_in_date} to {check_out_date}."
        elif "Rome" in location:
            return f"Found Colosseum View Hotel (ID: H301) - $190/night, Pantheon Suites (ID: H302) - $230/night for {guests} guests from {check_in_date} to {check_out_date}."
        else:
            return "No hotels found for the specified criteria. Please try a different location or dates."

    @Tool("Book Hotel")
    def book_hotel(hotel_id: str, user_details: str) -> str:
        """Books a hotel with the given ID for the specified user details.
        Returns a confirmation message with a booking ID.
        Example: 'Hotel H101 successfully booked for John Doe. Confirmation: HBOOK-12345.'
        """
        print(f"\n[Tool Call] Attempting to book hotel {hotel_id} for {user_details}")
        # Simulate booking success
        return f"Hotel {hotel_id} successfully booked for {user_details}. Confirmation: HBOOK-12345."

class FlightTools:
    @Tool("Search Flights")
    def search_flights(origin: str, destination: str, departure_date: str, return_date: str = None, passengers: int = 1) -> str:
        """Searches for available flights based on origin, destination, dates, and number of passengers.
        Returns a string summary of found flights including their IDs and prices.
        Example: 'Found Flight BA201 (ID: F101) - 2026-08-20 09:00, $150. Flight AF105 (ID: F102) - 2026-08-20 14:00, $180.'
        """
        print(f"\n[Tool Call] Searching flights from {origin} to {destination} on {departure_date} (Return: {return_date}) for {passengers} passengers")
        # Mock data based on route
        if "London" in origin and "Paris" in destination:
            return f"Found Flight BA201 (ID: F101) - {departure_date} 09:00, $150. Flight AF105 (ID: F102) - {departure_date} 14:00, $180. (Return: {return_date})"
        elif "New York" in origin and "London" in destination:
            return f"Found Flight UA001 (ID: F201) - {departure_date} 18:00, $600. Flight VS005 (ID: F202) - {departure_date} 20:00, $650. (Return: {return_date})"
        elif "Rome" in origin and "London" in destination:
            return f"Found Flight AZ300 (ID: F301) - {departure_date} 10:00, $250. Flight BA400 (ID: F302) - {departure_date} 16:00, $280. (Return: {return_date})"
        else:
            return "No flights found for the specified criteria. Please try a different route or dates."

    @Tool("Book Flight")
    def book_flight(flight_id: str, passenger_details: str) -> str:
        """Books a flight with the given ID for the specified passenger details.
        Returns a confirmation message with a booking ID.
        Example: 'Flight F101 successfully booked for Jane Doe. Confirmation: FBOOK-67890.'
        """
        print(f"\n[Tool Call] Attempting to book flight {flight_id} for {passenger_details}")
        # Simulate booking success
        return f"Flight {flight_id} successfully booked for {passenger_details}. Confirmation: FBOOK-67890."

# Instantiate tools
hotel_tools = HotelTools()
flight_tools = FlightTools()

# --- Agents --- #

# 1. Intent Router Agent: Determines the user's primary goal.
intent_router_agent = Agent(
    role='Intent Router',
    goal='Direct user requests to the correct booking sub-crew (Hotel, Flight, or Package).',
    backstory="""You are an expert AI assistant specializing in understanding user intent for travel bookings.
    Your primary role is to analyze the user's request and determine if it's for a hotel, a flight, or a combined package.
    You must be precise and ensure the request is routed to the most appropriate specialized agent or crew.
    If the intent is unclear, you should output 'clarify' so the system can ask for more information.
    """,
    verbose=True,
    allow_delegation=False, # This agent only routes, doesn't delegate tasks to other agents directly
    llm=llm
)

# 2. Hotel Booking Agents: Handle all aspects of hotel reservations.
hotel_search_agent = Agent(
    role='Hotel Search Specialist',
    goal='Find the best hotel options based on user criteria (location, dates, guests).',
    backstory="""You are a meticulous hotel search expert. You use advanced search tools to find hotels that perfectly match the user's location, dates, and guest count.
    You are skilled at extracting necessary details from natural language requests and asking clarifying questions if information is missing.
    You provide clear, concise options for the booking agent.
    """,
    tools=[hotel_tools.search_hotels],
    verbose=True,
    allow_delegation=True, # Can delegate to other agents if needed, though primarily uses tools
    llm=llm
)

hotel_booking_agent = Agent(
    role='Hotel Booking Agent',
    goal='Confirm and finalize hotel reservations after human approval.',
    backstory="""You are a diligent hotel booking agent. Once hotel options are identified and human approval is given,
    you use the booking tool to secure the reservation and provide confirmation details.
    You are responsible for ensuring all details are correct before booking and handling the human interaction for approval.
    """,
    tools=[hotel_tools.book_hotel],
    verbose=True,
    allow_delegation=False, # This agent performs the final booking, no further delegation
    llm=llm
)

# 3. Flight Booking Agents: Handle all aspects of flight reservations.
flight_search_agent = Agent(
    role='Flight Search Specialist',
    goal='Find the best flight options based on user criteria (origin, destination, dates, passengers).',
    backstory="""You are an expert in flight search, capable of finding optimal routes, dates, and prices.
    You use flight search tools to present clear options for the booking agent. You are adept at parsing complex travel requests and identifying missing information.
    """,
    tools=[flight_tools.search_flights],
    verbose=True,
    allow_delegation=True,
    llm=llm
)

flight_booking_agent = Agent(
    role='Flight Booking Agent',
    goal='Confirm and finalize flight reservations after human approval.',
    backstory="""You are a meticulous flight booking agent. Once flight options are identified and human approval is given,
    you use the booking tool to secure the reservation and provide confirmation details.
    You are responsible for ensuring all details are correct before booking and handling the human interaction for approval.
    """,
    tools=[flight_tools.book_flight],
    verbose=True,
    allow_delegation=False,
    llm=llm
)

# 4. Package Booking Agent: Orchestrates both flight and hotel bookings for package deals.
package_coordinator_agent = Agent(
    role='Package Booking Coordinator',
    goal='Orchestrate combined flight and hotel bookings, ensuring seamless package deals and human approval.',
    backstory="""You are the ultimate travel package expert. You coordinate between flight and hotel search/booking processes
    to create comprehensive travel packages. You ensure all components align and present a unified package for human approval.
    You are responsible for managing the entire package booking lifecycle, from search to final confirmation.
    You can leverage the capabilities of the specialized search and booking agents for individual components.
    """,
    verbose=True,
    allow_delegation=True, # Can delegate to search agents for flight and hotel components
    llm=llm
)

# --- Tasks --- #

# Task for Intent Routing
route_intent_task = Task(
    description="""Analyze the user's request: '{user_request}'.
    Determine if the user wants to book a 'hotel', a 'flight', or a 'package' (flight + hotel).
    Output ONLY the determined intent as a single word: 'hotel', 'flight', 'package', or 'clarify'.
    If the intent is unclear or insufficient information is provided to determine a specific booking type, output 'clarify'.
    Example: 'I need a hotel in Paris' -> 'hotel'
    Example: 'Book me a flight to London' -> 'flight'
    Example: 'I want a trip to Rome with flight and hotel' -> 'package'
    Example: 'Tell me about travel' -> 'clarify'
    """,
    agent=intent_router_agent,
    expected_output="A single word: 'hotel', 'flight', 'package', or 'clarify'."
)

# Tasks for Hotel Booking Crew
hotel_search_task = Task(
    description="""Search for hotels based on the user's request: '{user_request}'.
    Carefully extract location, check-in date, check-out date, and number of guests.
    If any information is missing, clearly state what is needed and ask the user for it.
    If all information is present, use the 'Search Hotels' tool.
    Provide a summary of available options, including hotel IDs and prices, to the Hotel Booking Agent.
    """,
    agent=hotel_search_agent,
    expected_output="A list of available hotel options with IDs and prices, or a request for more information from the user."
)

hotel_approval_task = Task(
    description="""Review the proposed hotel options from the Hotel Search Specialist.
    Present the best option(s) to the human for approval. Clearly state the hotel name, ID, price, and dates.
    Wait for human input (e.g., 'approve H101' or 'reject').
    If approved, extract the chosen hotel ID and confirm user details (e.g., 'John Doe, 1 adult').
    If rejected, inform the user and suggest alternative actions.
    """,
    agent=hotel_booking_agent, # This agent handles the human interaction for approval
    human_input=True, # This task requires human intervention
    expected_output="The chosen hotel ID and user details for booking, or a rejection message."
)

hotel_booking_task = Task(
    description="""Book the approved hotel using the provided hotel ID and user details from the approval task.
    Use the 'Book Hotel' tool to finalize the reservation.
    Generate a clear confirmation message including the booking ID and all relevant details.
    """,
    agent=hotel_booking_agent,
    expected_output="A confirmation message for the hotel booking, including booking ID and details."
)

# Tasks for Flight Booking Crew (NEW)
flight_search_task = Task(
    description="""Search for flights based on the user's request: '{user_request}'.
    Carefully extract origin, destination, departure date, return date (if applicable), and number of passengers.
    If any information is missing, clearly state what is needed and ask the user for it.
    If all information is present, use the 'Search Flights' tool.
    Provide a summary of available flight options, including flight IDs and prices, to the Flight Booking Agent.
    """,
    agent=flight_search_agent,
    expected_output="A list of available flight options with IDs and prices, or a request for more information from the user."
)

flight_approval_task = Task(
    description="""Review the proposed flight options from the Flight Search Specialist.
    Present the best option(s) to the human for approval. Clearly state the flight details (ID, route, dates, price).
    Wait for human input (e.g., 'approve F101' or 'reject').
    If approved, extract the chosen flight ID and confirm passenger details (e.g., 'Jane Doe, 1 adult').
    If rejected, inform the user and suggest alternative actions.
    """,
    agent=flight_booking_agent,
    human_input=True,
    expected_output="The chosen flight ID and passenger details for booking, or a rejection message."
)

flight_booking_task = Task(
    description="""Book the approved flight using the provided flight ID and passenger details from the approval task.
    Use the 'Book Flight' tool to finalize the reservation.
    Generate a clear confirmation message including the booking ID and all relevant details.
    """,
    agent=flight_booking_agent,
    expected_output="A confirmation message for the flight booking, including booking ID and details."
)

# Tasks for Package Booking Crew (NEW)
package_search_and_propose_task = Task(
    description="""Coordinate the search for both flights and hotels based on the user's package request: '{user_request}'.
    Carefully extract all necessary details for both flight (origin, destination, departure date, return date, passengers) and hotel (location, check-in date, check-out date, guests).
    If any information is missing for either component, clearly state what is needed and ask the user for it.
    If all information is present, delegate to the Flight Search Specialist and Hotel Search Specialist to find suitable options.
    Synthesize the results into a cohesive proposed package deal, including flight and hotel details, for human review.
    """,
    agent=package_coordinator_agent,
    expected_output="A proposed flight and hotel package summary, or a request for more information from the user."
)

package_approval_task = Task(
    description="""Review the proposed flight and hotel package from the Package Booking Coordinator.
    Present the complete package details (flight and hotel) to the human for approval. Clearly state all components, IDs, and total estimated cost.
    Wait for human input (e.g., 'approve package' or 'reject').
    If approved, extract the chosen flight ID, hotel ID, and user details for final booking.
    If rejected, inform the user and suggest alternative actions.
    """,
    agent=package_coordinator_agent,
    human_input=True,
    expected_output="The chosen flight ID, hotel ID, and user details for booking, or a rejection message."
)

package_final_booking_task = Task(
    description="""Finalize the approved flight and hotel package booking using the flight ID, hotel ID, and user details.
    This involves delegating to the Flight Booking Agent to book the flight and the Hotel Booking Agent to book the hotel.
    Ensure both bookings are successful.
    Generate a comprehensive confirmation message for the entire package, including both flight and hotel booking IDs.
    """,
    agent=package_coordinator_agent,
    expected_output="A comprehensive confirmation message for the flight and hotel package booking, including both booking IDs."
)

# --- Crews --- #

# Hotel Booking Crew: Handles the full lifecycle of a hotel reservation.
hotel_booking_crew = Crew(
    agents=[hotel_search_agent, hotel_booking_agent],
    tasks=[hotel_search_task, hotel_approval_task, hotel_booking_task],
    process=Process.sequential, # Tasks run in order
    verbose=True,
    manager_llm=llm # Manager LLM for task orchestration
)

# Flight Booking Crew: Handles the full lifecycle of a flight reservation.
flight_booking_crew = Crew(
    agents=[flight_search_agent, flight_booking_agent],
    tasks=[flight_search_task, flight_approval_task, flight_booking_task],
    process=Process.sequential,
    verbose=True,
    manager_llm=llm
)

# Package Booking Crew: Orchestrates both flight and hotel bookings.
# It includes all agents necessary for both flight and hotel components, as the coordinator will delegate.
package_booking_crew = Crew(
    agents=[
        package_coordinator_agent,
        hotel_search_agent, # Needed for hotel search within package
        hotel_booking_agent, # Needed for hotel booking within package
        flight_search_agent, # Needed for flight search within package
        flight_booking_agent # Needed for flight booking within package
    ],
    tasks=[
        package_search_and_propose_task,
        package_approval_task,
        package_final_booking_task
    ],
    process=Process.sequential,
    verbose=True,
    manager_llm=llm
)

# --- Main Router System --- #

class MultiServiceBookingSystem:
    def __init__(self):
        # The router crew's sole purpose is to determine the intent.
        self.router_crew = Crew(
            agents=[intent_router_agent],
            tasks=[route_intent_task],
            process=Process.sequential,
            verbose=True,
            manager_llm=llm
        )

    def run(self, user_request: str):
        print(f"\n==================================================")
        print(f"--- Initiating Multi-Service Booking System ---")
        print(f"--- User Request: '{user_request}' ---")
        print(f"==================================================\n")

        # First, route the intent
        intent_result = self.router_crew.kickoff(inputs={'user_request': user_request})
        intent = intent_result.strip().lower()

        print(f"\n--- Detected Intent: {intent.upper()} ---")
        print(f"--------------------------------------------------\n")

        final_output = ""
        if intent == 'hotel':
            print("\n--- Activating Hotel Booking Crew ---")
            final_output = hotel_booking_crew.kickoff(inputs={'user_request': user_request})
        elif intent == 'flight':
            print("\n--- Activating Flight Booking Crew ---")
            final_output = flight_booking_crew.kickoff(inputs={'user_request': user_request})
        elif intent == 'package':
            print("\n--- Activating Package Booking Crew ---")
            final_output = package_booking_crew.kickoff(inputs={'user_request': user_request})
        elif intent == 'clarify':
            final_output = "I'm sorry, I couldn't clearly understand your request. Could you please provide more details about what you'd like to book (hotel, flight, or a package)?"
        else:
            final_output = "An unexpected error occurred in intent routing. Please try again."

        print(f"\n==================================================")
        print(f"--- Final System Response ---")
        print(final_output)
        print(f"==================================================\n")
        return final_output

# --- Example Usage --- #
if __name__ == "__main__":
    booking_system = MultiServiceBookingSystem()

    # --- Scenario 1: Hotel Booking ---
    print("\n### Running Scenario 1: Hotel Booking ###")
    # User will be prompted for approval: Type 'approve H101' (or H102, H201, etc.) or 'reject'
    booking_system.run("I need a hotel in Paris for 2 adults from 2026-07-10 to 2026-07-15.")
    print("\n" + "="*80 + "\n")

    # --- Scenario 2: Flight Booking ---
    print("\n### Running Scenario 2: Flight Booking ###")
    # User will be prompted for approval: Type 'approve F101' (or F102, F201, etc.) or 'reject'
    booking_system.run("Book me a flight from London to Paris for one person on 2026-08-20, returning on 2026-08-25.")
    print("\n" + "="*80 + "\n")

    # --- Scenario 3: Package Booking ---
    print("\n### Running Scenario 3: Package Booking ###")
    # User will be prompted for approval: Type 'approve package' or 'reject'
    booking_system.run("I want a package trip to London from New York for 2 people, departing 2026-09-01 and returning 2026-09-07. Also need a hotel there.")
    print("\n" + "="*80 + "\n")

    # --- Scenario 4: Unclear Request ---
    print("\n### Running Scenario 4: Unclear Request ###")
    booking_system.run("Tell me about travel options.")
    print("\n" + "="*80 + "\n")

    # --- Scenario 5: Missing Information (Hotel) ---
    print("\n### Running Scenario 5: Missing Information (Hotel) ###")
    booking_system.run("I need a hotel for next week.")
    print("\n" + "="*80 + "\n")

    # --- Scenario 6: Missing Information (Flight) ---
    print("\n### Running Scenario 6: Missing Information (Flight) ###")
    booking_system.run("I want to fly to Rome.")
    print("\n" + "="*80 + "\n")

    # --- Scenario 7: Missing Information (Package) ---
    print("\n### Running Scenario 7: Missing Information (Package) ###")
    booking_system.run("I want a package deal to Rome.")
    print("\n" + "="*80 + "\n")
